# Шаг 06 — тот же агент как граф (LangGraph)

- **State** — общие данные; `MessagesState = {"messages": [...]}`, всё, что узел вернёт под `messages`, **добавляется** к истории;
- **Node** — функция: состояние → обновление (`agent`, `tools`);
- **Edge** — куда дальше; `tools_condition` — готовая проверка «есть tool_calls → tools, иначе → END».

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
from tools import list_files, read_file

TOOLS = [list_files, read_file]

SYSTEM = ("Ты ассистент по коду. Смотри реальные файлы через инструменты, "
          "ничего не выдумывай. Отвечай кратко.")

In [ ]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(TOOLS)


def agent(state: MessagesState) -> dict:
    """One model step: read the whole history, return the new message."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(TOOLS))              # runs the requested tools
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # → "tools" or END
builder.add_edge("tools", "agent")

graph = builder.compile()

Граф, который мы построили (стрелка `tools → agent` — это и есть цикл):

In [ ]:
from IPython.display import Image, display


def show_graph(g, xray: bool = False) -> None:
    """Draw a graph as a picture (rendered by mermaid.ink — needs internet), ASCII if offline."""
    try:
        display(Image(g.get_graph(xray=xray).draw_mermaid_png()))
    except Exception:
        print(g.get_graph(xray=xray).draw_ascii())   # note: ASCII does not draw back-edges


show_graph(graph)

Запускаем со `stream_mode="updates"` — видно, какой узел что сделал.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

def print_update(update: dict) -> None:
    """Show what each graph node did."""
    for node, data in update.items():
        for msg in data["messages"]:
            if node == "agent" and msg.tool_calls:
                for call in msg.tool_calls:
                    print(f"  [agent] → {call['name']}({call['args']})")
            elif node == "tools":
                print(f"  [tools] {msg.name} done")
            else:
                print("ai ›", msg.content)


history = [SystemMessage(SYSTEM)]


def ask(text: str) -> None:
    history.append(HumanMessage(text))
    for update in graph.stream({"messages": history}, stream_mode="updates"):
        print_update(update)
        for data in update.values():          # keep history for the next turn
            history.extend(data["messages"])

In [ ]:
ask("Что лежит в папке lecture?")

In [ ]:
ask("А что делает build.js в этой папке? Одной фразой.")

## Тот же агент одной строкой

`create_agent` из LangChain собирает ровно такой граф. Удобно в проде — но теперь понятно, что внутри.

In [ ]:
from langchain.agents import create_agent

prebuilt = create_agent(llm, TOOLS, system_prompt=SYSTEM)
show_graph(prebuilt)

In [ ]:
result = prebuilt.invoke({"messages": [{"role": "user", "content": "Сколько ноутбуков в текущей папке?"}]})
print(result["messages"][-1].content)

## Граф посложнее: роутер + подграф

Агент выше — это один цикл. Реальные графы обычно ветвятся. Соберём граф, который **сначала решает, что делать**, а потом отправляет запрос в нужную ветку:

```
START → router ─┬─ "chat" → chat ────────→ END
                └─ "code" → code_agent ──→ END
                             (весь агент выше — как ОДИН узел)
```

Новое здесь:
- **своё состояние**: кроме `messages` храним поле `route` — решение роутера;
- **роутер на structured output** (шаг 03): модель выбирает ветку из `Literal["chat", "code"]`;
- **условное ребро с явной картой** «значение → узел»;
- **подграф**: скомпилированный `graph` из начала ноутбука добавляется как обычный узел.

In [ ]:
from typing import Literal

from langgraph.graph import END
from pydantic import BaseModel, Field


class RouterState(MessagesState):
    route: str                      # filled by the router: "chat" or "code"


class Route(BaseModel):
    """Which branch should handle the user's request."""

    route: Literal["chat", "code"] = Field(
        description="code — about files, code or the current folder; chat — anything else")


router_llm = llm.with_structured_output(Route, method="function_calling")


def router(state: RouterState) -> dict:
    """Look at the last user message and pick a branch."""
    decision = router_llm.invoke([SystemMessage("Classify the user's request."),
                                  state["messages"][-1]])
    return {"route": decision.route}


def chat(state: RouterState) -> dict:
    """Plain answer from the model, no tools."""
    return {"messages": [llm.invoke(state["messages"])]}

Собираем: ветку выбирает функция `lambda state: state["route"]`, а словарь говорит, в какой узел идти для каждого значения. Узел `code_agent` — это целиком граф агента.

In [ ]:
builder2 = StateGraph(RouterState)
builder2.add_node("router", router)
builder2.add_node("chat", chat)
builder2.add_node("code_agent", graph)          # the whole agent from above as ONE node
builder2.add_edge(START, "router")
builder2.add_conditional_edges("router", lambda state: state["route"],
                               {"chat": "chat", "code": "code_agent"})
builder2.add_edge("chat", END)
builder2.add_edge("code_agent", END)
router_graph = builder2.compile()

show_graph(router_graph, xray=True)

`xray=True` раскрывает подграф — видно, что внутри `code_agent` тот же цикл agent ⇄ tools:

In [ ]:
show_graph(router_graph, xray=True)

Запускаем. `subgraphs=True` в `stream` показывает и шаги **внутри** подграфа (они с отступом).

In [ ]:
def run_router(question: str) -> None:
    inputs = {"messages": [SystemMessage(SYSTEM), HumanMessage(question)]}
    for namespace, update in router_graph.stream(inputs, stream_mode="updates", subgraphs=True):
        indent = "      " if namespace else ""      # namespace is non-empty inside the subgraph
        for node, data in update.items():
            if node == "router":
                print(f"[router] → {data['route']}")
            elif node == "code_agent":
                continue                             # already shown step by step from inside
            else:
                for msg in data["messages"]:
                    if getattr(msg, "tool_calls", None):
                        for call in msg.tool_calls:
                            print(f"{indent}[{node}] → {call['name']}({call['args']})")
                    elif node == "tools":
                        print(f"{indent}[tools] {msg.name} done")
                    else:
                        print(f"{indent}[{node}] {msg.content}")

In [ ]:
run_router("Привет! Объясни в одном предложении, что такое LangGraph.")

In [ ]:
run_router("Сколько ноутбуков лежит в текущей папке?")

**Итог.** Граф теперь сам выбирает путь, а подграфы позволяют собирать большие системы из маленьких проверенных кусков: агент из начала ноутбука стал просто узлом. Эта же идея — «агент как узел» — основа мультиагентных систем (занятие 3).